Semi-Isomorphism Test



ADD VISUALISER!!!!!!!!!

Choose examples for project report

Discuss to what extent each of these notions of isomorphism preserve synchronization properties of a DFA (if at all).


In [2]:
import networkx as nx

def is_semi_isomorphism(A1, A2):
    """
    Tests if two DFAs A1 = (Q1, Σ1, τ1, qs1, F1) and A2 = (Q2, Σ2, τ2, qs2, F2)
    are semi-isomorphic.

    Semi-isomorphism requires bijections f: Q1 → Q2 and g: Σ1 → Σ2 such that:
        τ2(f(q), g(a)) = f(τ1(q, a))  for all q ∈ Q1, a ∈ Σ1

    Start states and final states are ignored.
    We are guaranteed that in each DFA there is exactly one state
    that can reach all other states.
    """

    Q1, sigma1, tau1, qs1, F1 = A1
    Q2, sigma2, tau2, qs2, F2 = A2

    # If sizes differ, no bijection can exist
    if len(Q1) != len(Q2) or len(sigma1) != len(sigma2):
        return False, None, None

    # Build directed graphs for SCC analysis
    G1 = nx.MultiDiGraph()
    for q in Q1:
        for a in sigma1:
            G1.add_edge(q, tau1[(q, a)], label=a)

    G2 = nx.MultiDiGraph()
    for q in Q2:
        for a in sigma2:
            G2.add_edge(q, tau2[(q, a)], label=a)

    # Compute SCCs and their condensations
    sccs1 = list(nx.strongly_connected_components(G1))
    sccs2 = list(nx.strongly_connected_components(G2))

    C1 = nx.condensation(G1, scc=sccs1)
    C2 = nx.condensation(G2, scc=sccs2)

    # We assume that in each graph there is one state/node from which all other nodes are reachable.
    # From the SCCC functionality in NetworkX, in one single SCC, properties are inherited since they are all connected,
    # therefore, if there is one SCC which can reach every other SCC our node is in this SCC
    # We can then draw an semi-isomorphisms between these SCCs
    reaching_states1 = []
    reaching_states2 = []

    for node in C1.nodes():
        reachable = nx.descendants(C1, node)
        if len(reachable) == len(C1.nodes()) - 1:
            reaching_states1.extend(sccs1[node])

    for node in C2.nodes():
        reachable = nx.descendants(C2, node)
        if len(reachable) == len(C2.nodes()) - 1:
            reaching_states2.extend(sccs2[node])

    # Try every pairing of (root1, root2) as the initial guess of semi-isomorphism: 
    
    # We set up an isomorphism between the sets of states which reach every other state
    # for both A1 and A2, and that is the root of the semi-isomorphism.
    # From this we take pairs of (root1, root2) where root1 is a state
    # which can reach every state of A1 and similarly for root2.
    # We try to establish f(root1) = root2 as an initial mapping,
    # we then try to establish the rest of f and g by continuing the previous transitions.
    # If no combination is possible then no semi-isomorphism exists.
    # Since it is a semi-isomorphism we don't care about starting states, hence
    # the initial mapping of f isn't determined by qs — we try all initial root pairings.

    
    for root1 in reaching_states1:
        for root2 in reaching_states2:
            f = {root1: root2}
            queue = []
            for a in sigma1:
                queue.append((root1, a))
            result = BFS(f, {}, queue, tau1, tau2, sigma1, sigma2, Q1)
            if result != None:
                return True, result[0], result[1]

    return False, None, None

In [3]:
def BFS(f, g, queue, tau1, tau2, sigma1, sigma2, Q1):

    if queue == []:
        if (len(f) == len(Q1) and len(g) == len(sigma1)
                and len(set(f.values())) == len(Q1) and len(set(g.values())) == len(sigma1)):
            return f, g
        return None

    q, a = queue[0]
    rest = queue[1:]
    fq = f[q]
    q_next = tau1[(q, a)]

    if a in g and q_next in f:
        # Case 1: we know both f(q_next) and g(a) transformations
        if tau2[(fq, g[a])] != f[q_next]:
            return None
        return BFS(f, g, rest, tau1, tau2, sigma1, sigma2, Q1)

    elif a in g and q_next not in f:
        # Case 2: we know g(a) but don't know f(q_next)
        target = tau2[(fq, g[a])]
        if target in f.values():
            return None
        new_f = dict(f)
        new_f[q_next] = target
        for b in sigma1:
            rest.append((q_next, b))
        return BFS(new_f, g, rest, tau1, tau2, sigma1, sigma2, Q1)

    elif a not in g and q_next in f:
        # Case 3: we know f(q_next) but don't know g(a)
        target = f[q_next]
        for b in sigma2:
            if b not in g.values() and tau2[(fq, b)] == target:
                new_g = dict(g)
                new_g[a] = b
                result = BFS(f, new_g, rest, tau1, tau2, sigma1, sigma2, Q1)
                if result is not None:
                    return result
        return None

    else:
        # Case 4: we don't know g(a) or f(q_next)
        for b in sigma2:
            target = tau2[(fq, b)]
            if b not in g.values() and target not in f.values():
                new_f = dict(f)
                new_f[q_next] = target
                new_g = dict(g)
                new_g[a] = b
                new_queue = list(rest)
                for b2 in sigma1:
                    new_queue.append((q_next, b2))
                result = BFS(new_f, new_g, new_queue, tau1, tau2, sigma1, sigma2, Q1)
                if result is not None:
                    return result
        return None